# Zadanie 5 – Wieża Hanoi

## Czego wymaga zadanie?

Zaimplementować rekurencyjną procedurę `hanoi(FROM, AUXILIARY, TO, n)`, gdzie:
- `FROM`, `AUXILIARY`, `TO` – wieże reprezentowane jako **listy liczb naturalnych**
- `n` – liczba krążków do przeniesienia

Cel: przenieść wszystkie `n` krążków z wieży `FROM` na wieżę `TO`, używając `AUXILIARY` jako pomocniczej.

## Reprezentacja wież

Każda wieża to lista o długości `n`:  
- `lista[i] = k` oznacza, że krążek `k` leży na pozycji `i` od góry  
- `lista[i] = 0` oznacza **puste miejsce**  

**Stan początkowy** (n=4):  
```
F = [1, 2, 3, 4]   (krążek 1 na górze, 4 na dole)
A = [0, 0, 0, 0]   (pusta)
T = [0, 0, 0, 0]   (pusta)
```

**Stan końcowy**:
```
F = [0, 0, 0, 0]
A = [0, 0, 0, 0]
T = [1, 2, 3, 4]   (wszystkie krążki przeniesione)
```

## Algorytm rekurencyjny (zasada indukcji)

**Przypadek bazowy** (`n = 1`): przenieś jedyny krążek z FROM na TO.

**Przypadek rekurencyjny** (`n ≥ 2`) – zakładamy, że potrafimy przenieść `n-1` krążków:
1. Przenieś górne `n-1` krążków z `FROM` na `AUXILIARY` (używając `TO` jako pomocniczej)
2. Przenieś największy krążek `n` z `FROM` na `TO` (przypadek bazowy)
3. Przenieś `n-1` krążków z `AUXILIARY` na `TO` (używając `FROM` jako pomocniczej)

## Implementacja

In [ ]:
def znajdz_gorny_krazek(wieza):
    """Zwraca indeks i wartość najwyżej leżącego krążka (szuka od góry)."""
    for i, krazek in enumerate(wieza):
        if krazek != 0:
            return i, krazek
    return None, None  # wieża pusta


def znajdz_pierwsze_wolne(wieza):
    """Zwraca indeks pierwszego wolnego miejsca od dołu."""
    for i in range(len(wieza) - 1, -1, -1):
        if wieza[i] == 0:
            return i
    return None  # wieża pełna


def przesun_krazek(skad, dokad):
    """Przenosi górny krążek z wieży 'skad' na wieżę 'dokad'."""
    idx_skad, krazek = znajdz_gorny_krazek(skad)
    idx_dokad = znajdz_pierwsze_wolne(dokad)
    skad[idx_skad] = 0        # usuń z wieży źródłowej
    dokad[idx_dokad] = krazek  # postaw na wieży docelowej


def hanoi(FROM, AUXILIARY, TO, n):
    """
    Rekurencyjny algorytm wieży Hanoi.
    Przenosi n górnych krążków z wieży FROM na wieżę TO,
    używając AUXILIARY jako pomocniczej.
    """
    # Przypadek bazowy: jeden krążek – po prostu przesuń
    if n == 1:
        przesun_krazek(FROM, TO)
        return

    # Krok 1: przenieś n-1 krążków z FROM na AUXILIARY (TO jako pomocnicza)
    hanoi(FROM, TO, AUXILIARY, n - 1)

    # Krok 2: przenieś największy (n-ty) krążek z FROM na TO
    przesun_krazek(FROM, TO)

    # Krok 3: przenieś n-1 krążków z AUXILIARY na TO (FROM jako pomocnicza)
    hanoi(AUXILIARY, FROM, TO, n - 1)

## Wizualizacja stanów wież

In [ ]:
def wyswietl_wieze(F, A, T, tytul=""):
    """Wypisuje stan trzech wież w czytelnej formie."""
    if tytul:
        print(f"\n{tytul}")
    n = len(F)
    print(f"  {'F':^10} {'A':^10} {'T':^10}")
    print("  " + "-" * 32)
    for i in range(n):
        def fmt(v): return str(v) if v != 0 else "."
        print(f"  {fmt(F[i]):^10} {fmt(A[i]):^10} {fmt(T[i]):^10}")
    print()

In [ ]:
# Test dla n = 4 krążków
n = 4
F = list(range(1, n + 1))  # [1, 2, 3, 4] – krążek 1 na górze
A = [0] * n
T = [0] * n

wyswietl_wieze(F, A, T, "Stan POCZĄTKOWY:")

hanoi(F, A, T, n)

wyswietl_wieze(F, A, T, "Stan KOŃCOWY:")

# Weryfikacja: F i A puste, T = [1, 2, 3, 4]
oczekiwane_T = list(range(1, n + 1))
if F == [0]*n and A == [0]*n and T == oczekiwane_T:
    print("Poprawne przeniesienie wszystkich krążków!")
else:
    print("BLAD – niepoprawny stan końcowy!")

## Wersja z rejestrowaniem ruchów

In [ ]:
ruchy = []

def hanoi_z_logiem(FROM, AUXILIARY, TO, n, nazwa_from, nazwa_aux, nazwa_to):
    """Jak hanoi(), ale rejestruje każdy ruch (który krążek, skąd, dokąd)."""
    if n == 1:
        _, krazek = znajdz_gorny_krazek(FROM)
        przesun_krazek(FROM, TO)
        ruchy.append((krazek, nazwa_from, nazwa_to))
        return

    hanoi_z_logiem(FROM, TO, AUXILIARY, n - 1, nazwa_from, nazwa_to, nazwa_aux)
    _, krazek = znajdz_gorny_krazek(FROM)
    przesun_krazek(FROM, TO)
    ruchy.append((krazek, nazwa_from, nazwa_to))
    hanoi_z_logiem(AUXILIARY, FROM, TO, n - 1, nazwa_aux, nazwa_from, nazwa_to)


# Uruchom dla n=3 i wypisz wszystkie ruchy
n = 3
F = list(range(1, n + 1))
A = [0] * n
T = [0] * n
ruchy.clear()

hanoi_z_logiem(F, A, T, n, "F", "A", "T")

print(f"Sekwencja ruchów dla n={n} krążków ({len(ruchy)} ruchów):")
for i, (krazek, skad, dokad) in enumerate(ruchy, 1):
    print(f"  Ruch {i:2d}: krążek {krazek} z {skad} → {dokad}")

## Liczba ruchów i złożoność

In [ ]:
def liczba_ruchow_hanoi(n):
    """Optymalna liczba ruchów dla n krążków = 2^n - 1."""
    return 2**n - 1

print(f"{'n':>5} {'2^n - 1':>10} {'opis'}")
print("-" * 40)
for n in range(1, 16):
    ruchy_n = liczba_ruchow_hanoi(n)
    print(f"{n:>5} {ruchy_n:>10}")

print()
print("Dla n=64 (legendarne 64 złote krążki):")
print(f"  Liczba ruchów: {2**64 - 1:,}")
print(f"  Przy 1 ruchu/sekundę: {(2**64-1) / (3600*24*365.25):.2e} lat")

## Weryfikacja dla różnych n

In [ ]:
print(f"{'n':>4}  {'oczekiwane T':>20}  {'wynik poprawny?':>16}")
print("-" * 45)
for n in range(1, 8):
    F = list(range(1, n + 1))
    A = [0] * n
    T = [0] * n
    hanoi(F, A, T, n)
    oczekiwane = list(range(1, n + 1))
    ok = (F == [0]*n and A == [0]*n and T == oczekiwane)
    print(f"{n:>4}  {str(oczekiwane):>20}  {'OK' if ok else 'BLAD':>16}")

## Podsumowanie

| Element | Opis |
|---|---|
| **Funkcja** | `hanoi(FROM, AUXILIARY, TO, n)` |
| **Wieże** | listy liczb naturalnych; `0` = puste miejsce |
| **Przypadek bazowy** | `n == 1` → przesuń jeden krążek |
| **Rekurencja** | 3 kroki: n-1 na AUX, największy na TO, n-1 z AUX na TO |
| **Liczba ruchów** | $2^n - 1$ (minimalna i zarazem uzyskiwana) |
| **Złożoność** | $O(2^n)$ – wykładnicza, nieunikniona |